# READING THE CSV_FILE

In [117]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder

df=pd.read_csv("housing.csv")
df.info()
print("missing:", df.isnull().sum().sum(), "  duplicates:", df.duplicated().sum())
# filtering=df[df['ocean_proximity']=="NEAR OCEAN"]
# print(filtering)
print(df.head(10))

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB
missing: 207   duplicates: 0
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24

# DETRMINE THE TARGET FEATURE

In [118]:
target="median_house_value"
x=df.drop(columns=target)
y = df[target]

In [119]:
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr
low, high = iqr_bounds(df["median_house_value"])
mask = (df["median_house_value"] < low) | (df["median_house_value"] > high)
print(mask.sum())


1071


# DEALING WITH MISSING VALUES

In [120]:
df['total_bedrooms']=df['total_bedrooms'].fillna(df['total_bedrooms'].median())
print(df.isnull().sum())

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


# FINDING THE CORRELATION WITH EACH FEATURE

In [121]:
features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 
            'total_bedrooms', 'population', 'households', 'median_income']
corr_with_target = df[features].corrwith(df['median_house_value']).abs().sort_values(ascending=False)
print(corr_with_target)

median_income         0.688075
latitude              0.144160
total_rooms           0.134153
housing_median_age    0.105623
households            0.065843
total_bedrooms        0.049457
longitude             0.045967
population            0.024650
dtype: float64


# FEATURE SELECTION

In [122]:
df=df.drop(columns=['population'])
print(df.head(10))

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   
5    -122.25     37.85                52.0        919.0           213.0   
6    -122.25     37.84                52.0       2535.0           489.0   
7    -122.25     37.84                52.0       3104.0           687.0   
8    -122.26     37.84                42.0       2555.0           665.0   
9    -122.25     37.84                52.0       3549.0           707.0   

   households  median_income  median_house_value ocean_proximity  
0       126.0         8.3252            452600.0        NEAR BAY  
1      1138.0         8.3014            

In [123]:
print(df['ocean_proximity'].value_counts())

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64


# ENCODING CATEGORICAL FEATURES

In [124]:
ohe = OneHotEncoder(drop="first", sparse_output=False)
encoded_array = ohe.fit_transform(df[["ocean_proximity"]])
encoded_columns = ohe.get_feature_names_out(["ocean_proximity"])
df_encoded = pd.DataFrame(encoded_array, columns=encoded_columns, index=df.index)
df = pd.concat([df.drop(columns=["ocean_proximity"]), df_encoded], axis=1)
print(df.head())
print(df.info())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   households  median_income  median_house_value  ocean_proximity_INLAND  \
0       126.0         8.3252            452600.0                     0.0   
1      1138.0         8.3014            358500.0                     0.0   
2       177.0         7.2574            352100.0                     0.0   
3       219.0         5.6431            341300.0                     0.0   
4       259.0         3.8462            342200.0                     0.0   

   ocean_proximity_ISLAND  ocean_proximity_NEAR BAY  \
0                     0.0            